In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import pandas as pd
from IPython.display import display
from IPython.display import set_matplotlib_formats
from collections import defaultdict
from difflib import SequenceMatcher
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
import csv

In [ ]:
base_path = "/content/drive/MyDrive/Colab Notebooks/Memoire_M1/Evaluation_DARIEN"

In [ ]:
spacy_ref = {
    "REF": f"{base_path}/DARIEN/REF/DARIEN_Le-voleur_REF.txt_spacy-lg-3.7.5.json",
    "TESSERACT": f"{base_path}/DARIEN/TESSERACT/DARIEN_Le-voleur_tesseract0.3.10.txt_spacy-lg-3.7.5.json",
    "KRAKEN": f"{base_path}/DARIEN/KRAKEN/DARIEN_Le-voleur_kraken4.3.13.dev25.txt_spacy-lg-3.7.5.json"
}

In [ ]:
model_dirs = {
    "initial": f"{base_path}/resultats_complet_initial",
    "enrichi": f"{base_path}/resultats_complet_enrichi",
    "quaero_fusioné": f"{base_path}/resultats_complet_enrichi_quaero_fusioné",
    "enrichi_80-20": f"{base_path}/resultats_complet_enrichi_80-20",
}

In [ ]:
model_files = {
    modele: {
        "REF": f"{model_dirs[modele]}/DARIEN_Le-voleur_REF_tok2vec_{modele}.json",
        "TESSERACT": f"{model_dirs[modele]}/DARIEN_Le-voleur_tesseract0.3.10_tok2vec_{modele}.json",
        "KRAKEN": f"{model_dirs[modele]}/DARIEN_Le-voleur_kraken4.3.13.dev25_tok2vec_{modele}.json"
    } for modele in model_dirs
}

**1. Création d'un CSV qui évalue nos modèles en les comparant avec spacy ("comparaison_metrics.csv")**

In [ ]:
def normaliser_label(label):
    if isinstance(label, list):
        label = label[0]
    if isinstance(label, str):
        label = label.strip()
        if label.startswith("LOC"):
            return "LOC"
    return label

def charger_entites(json_path):
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, dict):
            entites = data.values()
        elif isinstance(data, list):
            entites = data
        else:
            return set()
        resultat = set()
        for ent in entites:
            labels = ent.get("label", "")
            if isinstance(labels, str):
                labels = labels.split(",")
            elif isinstance(labels, list):
                pass
            else:
                labels = []
            for l in labels:
                resultat.add((
                    ent.get("text", "").strip(),
                    tuple(ent.get("jalons", [])),
                    normaliser_label(l)
                ))
        return resultat
    except Exception as e:
        print(f"Erreur chargement {json_path} : {e}")
        return set()

def calculer_scores(reference, prediction):
    vp = len(reference & prediction)
    fp = len(prediction - reference)
    fn = len(reference - prediction)
    precision = vp / (vp + fp) if (vp + fp) > 0 else 0
    rappel = vp / (vp + fn) if (vp + fn) > 0 else 0
    f1 = 2 * precision * rappel / (precision + rappel) if (precision + rappel) > 0 else 0
    return vp, fp, fn, round(precision, 4), round(rappel, 4), round(f1, 4)

In [ ]:
resultats = []
for modele, chemins in model_files.items():
    for ocr, chemin_modele in chemins.items():
        chemin_spacy = spacy_ref[ocr]
        ref_ents = charger_entites(chemin_spacy)
        pred_ents = charger_entites(chemin_modele)
        vp, fp, fn, precision, rappel, f1 = calculer_scores(ref_ents, pred_ents)
        resultats.append({
            "OCR": ocr,
            "Modèle": modele,
            "VP": vp,
            "FP": fp,
            "FN": fn,
            "Précision": precision,
            "Rappel": rappel,
            "F1-score": f1
        })

df = pd.DataFrame(resultats)
os.makedirs(f"{base_path}/metrics_comparaison_spacy", exist_ok=True)
df.to_csv(f"{base_path}/metrics_comparaison_spacy/comparaison_metrics.csv", index=False)
display(df)

**2. Comparaison avec tolérance de +/- caractères**
On va maintenant tester en étant moins discriminant:
Deux entités (référence vs prédiction) seront considérées comme équivalentes (VP) si :

Les labels sont identiques (après normalisation si nécessaire),

Les entités se chevauchent suffisamment :

Une intersection de plus de 50 % des caractères sur l'une ou l'autre des entités (IoU asymétrique),

Et une proximité de ±5 caractères entre leurs indices de début ou fin.

In [ ]:
def charger_entites_fuzzy(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    entites = []
    for ent in data.values():
        texte = ent.get("text", "")
        label = ent.get("label", "").strip()
        label = normaliser_label(label)
        debut, fin = ent.get("jalons", [None, None])
        if None not in (texte, label, debut, fin):
            entites.append({"text": texte, "label": label, "start": debut, "end": fin})
    return entites

def entites_equivalentes(ent1, ent2, tolerance=5):
    if normaliser_label(ent1["label"]) != normaliser_label(ent2["label"]):
        return False
    distance_debut = abs(ent1["start"] - ent2["start"])
    distance_fin = abs(ent1["end"] - ent2["end"])
    if distance_debut <= tolerance and distance_fin <= tolerance:
        # Vérifier le chevauchement de texte (mesure de similarité sur le texte)
        sm = SequenceMatcher(None, ent1["text"].lower(), ent2["text"].lower())
        if sm.ratio() > 0.5:
            return True
    return False

def evaluer_avec_tolerance(entites_ref, entites_modele):
    vp = 0
    entites_trouvees = set()
    for ref in entites_ref:
        for i, pred in enumerate(entites_modele):
            if i not in entites_trouvees and entites_equivalentes(ref, pred):
                vp += 1
                entites_trouvees.add(i)
                break
    fp = len(entites_modele) - vp
    fn = len(entites_ref) - vp
    precision = vp / (vp + fp) if (vp + fp) else 0
    rappel = vp / (vp + fn) if (vp + fn) else 0
    f1 = 2 * precision * rappel / (precision + rappel) if (precision + rappel) else 0
    return vp, fp, fn, precision, rappel, f1

In [ ]:
output_path = f"{base_path}/metrics_comparaison_spacy/resultats_comparaison_tolerance.csv"

resultats = []
for modele, fichiers in model_files.items():
    for ocr, chemin_model in fichiers.items():
        chemin_spacy = spacy_ref[ocr]
        try:
            ref = charger_entites_fuzzy(chemin_spacy)
            pred = charger_entites_fuzzy(chemin_model)
            vp, fp, fn, p, r, f1 = evaluer_avec_tolerance(ref, pred)
            resultats.append({
                "OCR": ocr,
                "Modèle": modele,
                "VP": vp,
                "FP": fp,
                "FN": fn,
                "Précision": round(p, 3),
                "Rappel": round(r, 3),
                "F1-score": round(f1, 3),
                "Nb_ENT_model": len(pred),
                "Nb_ENT_spacy": len(ref)
            })
        except Exception as e:
            print(f"Erreur : {chemin_model} | {e}")

df = pd.DataFrame(resultats)
df = df.sort_values(by=["OCR", "Modèle"])
matplotlib.rcParams.update({'font.size': 9})
df.to_csv(output_path, index=False)
display(df)

**3. Création de CSV afin d'observer le contexte ainsi que les différences de labellisation entre nos modèles et Spacy**

In [ ]:
def extraire_contexte(texte, debut, fin, taille=30):
    gauche = texte[max(0, debut - taille):debut].replace("\n", " ")
    droite = texte[fin:fin + taille].replace("\n", " ")
    return gauche.strip(), droite.strip()

def entites_equivalentes_fuzzy(ent1, ent2, tolerance=5):
    if ent1["label"] != ent2["label"]:
        return False
    distance_debut = abs(ent1["start"] - ent2["start"])
    distance_fin = abs(ent1["end"] - ent2["end"])
    if distance_debut <= tolerance and distance_fin <= tolerance:
        sm = SequenceMatcher(None, ent1["text"].lower(), ent2["text"].lower())
        return sm.ratio() > 0.5
    return False

def evaluer_avec_tolerance_et_labels(entites_ref, entites_modele):
    vp_ex, fp_ex, fn_ex = [], [], []
    deja_trouvees = set()

    for ref in entites_ref:
        trouve = False
        for i, pred in enumerate(entites_modele):
            if i not in deja_trouvees and entites_equivalentes_fuzzy(ref, pred):
                deja_trouvees.add(i)
                trouve = True
                vp_ex.append({
                    "text": ref["text"],
                    "start": ref["start"],
                    "end": ref["end"],
                    "label_spacy": ref["label"],
                    "label_modele": pred["label"]
                })
                break
        if not trouve:
            fn_ex.append({
                "text": ref["text"],
                "start": ref["start"],
                "end": ref["end"],
                "label_spacy": ref["label"],
                "label_modele": None
            })

    for i, pred in enumerate(entites_modele):
        if i not in deja_trouvees:
            # Chercher un label spacy proche (si c'était juste une erreur d'étiquette)
            label_spacy = None
            for ref in entites_ref:
                if abs(ref["start"] - pred["start"]) <= 5 and abs(ref["end"] - pred["end"]) <= 5:
                    ratio = SequenceMatcher(None, ref["text"].lower(), pred["text"].lower()).ratio()
                    if ratio > 0.5:
                        label_spacy = ref["label"]
                        break

            fp_ex.append({
                "text": pred["text"],
                "start": pred["start"],
                "end": pred["end"],
                "label_spacy": label_spacy,
                "label_modele": pred["label"]
            })

    return vp_ex, fp_ex, fn_ex

In [ ]:
textes_paths = {
    "REF": f"{base_path}/DARIEN/REF/DARIEN_Le-voleur_REF.txt",
    "TESSERACT": f"{base_path}/DARIEN/TESSERACT/DARIEN_Le-voleur_tesseract0.3.10.txt",
    "KRAKEN": f"{base_path}/DARIEN/KRAKEN/DARIEN_Le-voleur_kraken4.3.13.dev25.txt"
}

In [ ]:
output_dir = "/content/drive/MyDrive/Colab Notebooks/Memoire_M1/Evaluation_DARIEN/metrics_comparaison_spacy/analyse_erreurs_final"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
resultats = []
for modele, fichiers in model_files.items():
    for ocr, chemin_model in fichiers.items():
        chemin_spacy = spacy_ref[ocr]
        try:
            with open(chemin_spacy, "r", encoding="utf-8") as f:
                entites_ref = json.load(f).values()
                entites_ref = [
                    {"text": e["text"], "start": e["jalons"][0], "end": e["jalons"][1], "label": normaliser_label(e["label"])}
                    for e in entites_ref if e.get("text") and e.get("jalons") and e.get("label")
                ]
            with open(chemin_model, "r", encoding="utf-8") as f:
                entites_pred = json.load(f).values()
                entites_pred = [
                    {"text": e["text"], "start": e["jalons"][0], "end": e["jalons"][1], "label": normaliser_label(e["label"])}
                    for e in entites_pred if e.get("text") and e.get("jalons") and e.get("label")
                ]

            vp_ex, fp_ex, fn_ex = evaluer_avec_tolerance_et_labels(entites_ref, entites_pred)

            resultats.append({
                "OCR": ocr,
                "Modèle": modele,
                "VP_ex": vp_ex,
                "FP_ex": fp_ex,
                "FN_ex": fn_ex
            })

        except Exception as e:
            print(f"Erreur : {chemin_model} | {e}")

In [ ]:
for res in resultats:
    ocr = res["OCR"]
    modele = res["Modèle"]

    try:
        with open(textes_paths[ocr], "r", encoding="utf-8") as f:
            texte = f.read()
    except Exception as e:
        print(f"Erreur lecture fichier texte pour {ocr} : {e}")
        continue

    for categorie in ["VP", "FP", "FN"]:
        exemples = res.get(f"{categorie}_ex", [])
        lignes = []

        for exemple in exemples:
            start = exemple["start"]
            end = exemple["end"]
            gauche = texte[max(0, start - 30):start]
            droit = texte[end:end + 30]

            ligne = {
                "Gauche": gauche,
                "Début": start,
                "Texte": exemple["text"],
                "Label_modele": exemple.get("label_modele"),
                "Label_spacy": exemple.get("label_spacy"),
                "Fin": end,
                "Droit": droit
            }
            lignes.append(ligne)

        if lignes:
            df = pd.DataFrame(lignes)
            nom_fichier = f"{ocr}_{modele}_{categorie}.csv"
            chemin_csv = os.path.join(output_dir, nom_fichier)
            df.to_csv(chemin_csv, index=False, encoding="utf-8")